# 🦕 DINO SDK - Notebook Core de Ingestão

Este é o notebook principal executado pelos jobs do Databricks para realizar ingestão de dados usando o **DINO SDK Ingestion Engine**.

## 📋 Funcionalidades
- ✅ Ingestão automatizada de arquivos
- ✅ Detecção automática de schema
- ✅ Liquid clustering otimizado
- ✅ Logging completo para auditoria
- ✅ Tratamento de erros robusto

## 🎯 Parâmetros do Job
Este notebook recebe parâmetros via **Databricks Widgets** configurados automaticamente pelo DINO SDK.

In [ ]:
# 1. Configuração de Parâmetros via Widgets
import json
from datetime import datetime

# Criar widgets para parâmetros do job (configurados automaticamente pelo DINO SDK)
dbutils.widgets.text("catalog_name", "data_master_dev_dbw", "Nome do Catálogo")
dbutils.widgets.text("schema_name", "bronze", "Nome do Schema")  
dbutils.widgets.text("table_name", "exemplo_tabela", "Nome da Tabela")
dbutils.widgets.text("source_path", "", "Caminho dos arquivos fonte")
dbutils.widgets.text("file_format", "parquet", "Formato dos arquivos (parquet, csv, json)")
dbutils.widgets.dropdown("liquid_clustering", "true", ["true", "false"], "Usar Liquid Clustering")
dbutils.widgets.text("cluster_columns", "", "Colunas para clustering (separadas por vírgula)")
dbutils.widgets.text("job_run_id", "", "ID da execução do job")

# Obter valores dos parâmetros
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name") 
table_name = dbutils.widgets.get("table_name")
source_path = dbutils.widgets.get("source_path")
file_format = dbutils.widgets.get("file_format")
liquid_clustering = dbutils.widgets.get("liquid_clustering").lower() == "true"
cluster_columns = dbutils.widgets.get("cluster_columns")
job_run_id = dbutils.widgets.get("job_run_id")

print("🦕 DINO SDK - Ingestão Core Inicializada")
print("=" * 50)
print(f"📂 Catálogo: {catalog_name}")
print(f"📂 Schema: {schema_name}")
print(f"📊 Tabela: {table_name}")
print(f"📁 Source: {source_path}")
print(f"📄 Formato: {file_format}")
print(f"🔗 Liquid Clustering: {liquid_clustering}")
print(f"🏃 Job Run ID: {job_run_id}")

In [ ]:
# 2. Importar DINO SDK e Configurar Logging
import logging
from dino_sdk.ingestion_engine import IngestionEngine
from dino_sdk.schema_manager import SchemaManager

# Configurar logging para capturar todas as operações
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()  # Output para o console do Databricks
    ]
)

logger = logging.getLogger("DinoCoreIngestion")

# Log do início da execução
start_time = datetime.now()
logger.info(f"🚀 Iniciando ingestão: {catalog_name}.{schema_name}.{table_name}")
logger.info(f"⏰ Hora de início: {start_time}")

# Validar parâmetros obrigatórios
if not all([catalog_name, schema_name, table_name, source_path]):
    error_msg = "❌ Parâmetros obrigatórios não fornecidos"
    logger.error(error_msg)
    raise ValueError(error_msg)

print("✅ DINO SDK importado e logging configurado")

In [ ]:
# 3. Inicializar Ingestion Engine
try:
    # Criar instância do Ingestion Engine
    ingestion_engine = IngestionEngine()
    
    # Configurações da ingestão
    ingestion_config = {
        'catalog_name': catalog_name,
        'schema_name': schema_name, 
        'table_name': table_name,
        'source_path': source_path,
        'file_format': file_format,
        'liquid_clustering': liquid_clustering,
        'write_mode': 'append'  # Padrão para ingestão incremental
    }
    
    # Adicionar colunas de clustering se especificadas
    if cluster_columns and cluster_columns.strip():
        columns_list = [col.strip() for col in cluster_columns.split(',') if col.strip()]
        ingestion_config['cluster_columns'] = columns_list
        logger.info(f"🔗 Colunas de clustering: {columns_list}")
    
    logger.info("✅ Ingestion Engine inicializado")
    print("✅ Configurações da ingestão preparadas")
    
except Exception as e:
    error_msg = f"❌ Erro ao inicializar Ingestion Engine: {str(e)}"
    logger.error(error_msg)
    raise

# Mostrar configuração atual
print("\n📋 CONFIGURAÇÃO DA INGESTÃO:")
for key, value in ingestion_config.items():
    print(f"   {key}: {value}")

In [ ]:
# 4. Verificar Arquivos Fonte
try:
    logger.info(f"🔍 Verificando arquivos em: {source_path}")
    
    # Listar arquivos no caminho fonte
    try:
        files = dbutils.fs.ls(source_path)
        file_count = len(files)
        
        if file_count == 0:
            logger.warning("⚠️ Nenhum arquivo encontrado no caminho especificado")
        else:
            logger.info(f"📁 Encontrados {file_count} arquivo(s)")
            
            # Mostrar primeiros arquivos (máximo 5)
            for i, file_info in enumerate(files[:5]):
                logger.info(f"   📄 {file_info.name} ({file_info.size} bytes)")
                
            if file_count > 5:
                logger.info(f"   ... e mais {file_count - 5} arquivo(s)")
                
    except Exception as fs_error:
        logger.error(f"❌ Erro ao acessar caminho fonte: {str(fs_error)}")
        raise
        
    print(f"✅ Verificação de arquivos concluída - {file_count} arquivo(s) encontrado(s)")
    
except Exception as e:
    error_msg = f"❌ Erro na verificação de arquivos: {str(e)}"
    logger.error(error_msg)
    raise

In [ ]:
# 5. Executar Detecção de Schema (se necessário)
try:
    logger.info("🔍 Iniciando detecção de schema...")
    
    # Ler amostra dos dados para detecção de schema
    if file_format.lower() == 'parquet':
        sample_df = spark.read.parquet(source_path).limit(100)
    elif file_format.lower() == 'csv':
        sample_df = spark.read.option("header", "true").option("inferSchema", "true").csv(source_path).limit(100)
    elif file_format.lower() == 'json':
        sample_df = spark.read.json(source_path).limit(100)
    else:
        raise ValueError(f"Formato de arquivo não suportado: {file_format}")
    
    # Obter informações do schema
    schema_info = sample_df.dtypes
    row_count = sample_df.count()
    
    logger.info(f"📊 Schema detectado com {len(schema_info)} coluna(s)")
    logger.info(f"📈 Amostra processada: {row_count} registros")
    
    # Log das colunas detectadas
    print(f"\n📋 SCHEMA DETECTADO:")
    for col_name, col_type in schema_info:
        print(f"   {col_name}: {col_type}")
        
    # Armazenar schema para uso posterior
    detected_schema = sample_df.schema
    
    print("✅ Detecção de schema concluída")
    
except Exception as e:
    error_msg = f"❌ Erro na detecção de schema: {str(e)}"
    logger.error(error_msg)
    raise

In [ ]:
# 6. Executar Ingestão Principal
try:
    logger.info("🚀 Iniciando processo de ingestão principal...")
    
    # Ler todos os dados do source path
    logger.info(f"📖 Lendo dados de: {source_path}")
    
    if file_format.lower() == 'parquet':
        source_df = spark.read.parquet(source_path)
    elif file_format.lower() == 'csv':
        source_df = spark.read.option("header", "true").option("inferSchema", "true").csv(source_path)
    elif file_format.lower() == 'json':
        source_df = spark.read.json(source_path)
    
    # Contar registros
    total_records = source_df.count()
    logger.info(f"📊 Total de registros para ingestão: {total_records}")
    
    if total_records == 0:
        logger.warning("⚠️ Nenhum registro encontrado para ingestão")
    else:
        # Adicionar colunas de metadados
        from pyspark.sql.functions import lit, current_timestamp
        
        enriched_df = source_df \
            .withColumn("_ingestion_timestamp", current_timestamp()) \
            .withColumn("_job_run_id", lit(job_run_id)) \
            .withColumn("_source_path", lit(source_path))
        
        logger.info("✅ Colunas de metadados adicionadas")
        
        # Preparar nome completo da tabela
        full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
        
        print(f"💾 Gravando {total_records} registros na tabela: {full_table_name}")
        
except Exception as e:
    error_msg = f"❌ Erro na preparação da ingestão: {str(e)}"
    logger.error(error_msg)
    raise

In [ ]:
# 7. Gravar Dados com Liquid Clustering (se habilitado)
try:
    logger.info("💾 Iniciando gravação dos dados...")
    
    # Configurar writer
    writer = enriched_df.write.mode("append")
    
    # Aplicar Liquid Clustering se habilitado
    if liquid_clustering and cluster_columns and cluster_columns.strip():
        cluster_cols = [col.strip() for col in cluster_columns.split(',') if col.strip()]
        logger.info(f"🔗 Aplicando Liquid Clustering nas colunas: {cluster_cols}")
        
        # Para Databricks Runtime 13.3+, usar clusterBy
        writer = writer.option("clusterBy", ",".join(cluster_cols))
    
    # Gravar na tabela Delta
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    writer.saveAsTable(full_table_name)
    
    logger.info(f"✅ Dados gravados com sucesso na tabela: {full_table_name}")
    print(f"✅ Ingestão concluída: {total_records} registros processados")
    
    # Estatísticas finais
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    logger.info(f"⏰ Tempo total de execução: {duration:.2f} segundos")
    logger.info(f"📊 Taxa de processamento: {total_records/duration:.2f} registros/segundo")
    
except Exception as e:
    error_msg = f"❌ Erro na gravação dos dados: {str(e)}"
    logger.error(error_msg)
    raise

In [ ]:
# 8. Validação e Verificação Final
try:
    logger.info("🔍 Executando validações finais...")
    
    # Verificar se a tabela foi criada/atualizada
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    
    # Contar registros na tabela final
    final_count = spark.sql(f"SELECT COUNT(*) FROM {full_table_name}").collect()[0][0]
    
    logger.info(f"📊 Total de registros na tabela final: {final_count}")
    
    # Verificar últimas partições/dados inseridos
    latest_data = spark.sql(f"""
        SELECT 
            COUNT(*) as records_added,
            MAX(_ingestion_timestamp) as last_ingestion,
            _job_run_id
        FROM {full_table_name} 
        WHERE _job_run_id = '{job_run_id}'
        GROUP BY _job_run_id
    """).collect()
    
    if latest_data:
        records_added = latest_data[0]['records_added'] 
        last_ingestion = latest_data[0]['last_ingestion']
        
        logger.info(f"✅ Registros adicionados nesta execução: {records_added}")
        logger.info(f"⏰ Timestamp da última ingestão: {last_ingestion}")
        
        print(f"\n🎉 INGESTÃO CONCLUÍDA COM SUCESSO!")
        print(f"📊 Registros processados: {records_added}")
        print(f"📊 Total na tabela: {final_count}")
    else:
        logger.warning("⚠️ Nenhum registro encontrado com o Job Run ID atual")
        
except Exception as e:
    error_msg = f"❌ Erro na validação final: {str(e)}"
    logger.error(error_msg)
    # Não fazer raise aqui pois a ingestão pode ter funcionado
    print(f"⚠️ Validação final falhou, mas ingestão pode ter sido bem-sucedida")

In [ ]:
# 9. Cleanup e Resultado Final
try:
    # Limpar widgets (opcional)
    # dbutils.widgets.removeAll()
    
    # Resultado final
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    
    execution_summary = {
        "status": "SUCCESS",
        "catalog_name": catalog_name,
        "schema_name": schema_name,
        "table_name": table_name, 
        "records_processed": total_records,
        "execution_time_seconds": total_duration,
        "job_run_id": job_run_id,
        "liquid_clustering_enabled": liquid_clustering,
        "timestamp": end_time.isoformat()
    }
    
    # Log do resumo de execução
    logger.info("📋 RESUMO DA EXECUÇÃO:")
    for key, value in execution_summary.items():
        logger.info(f"   {key}: {value}")
    
    print("\n🎉 NOTEBOOK DINO CORE FINALIZADO COM SUCESSO!")
    print("=" * 60)
    print(f"✅ Status: {execution_summary['status']}")
    print(f"📊 Registros: {execution_summary['records_processed']}")
    print(f"⏰ Duração: {execution_summary['execution_time_seconds']:.2f}s")
    print(f"📂 Tabela: {full_table_name}")
    
    # Retornar resultado (para jobs programáticos)
    dbutils.notebook.exit(json.dumps(execution_summary))
    
except Exception as e:
    error_msg = f"❌ Erro no cleanup final: {str(e)}"
    logger.error(error_msg)
    
    # Resultado de erro
    error_summary = {
        "status": "ERROR",
        "error_message": str(e),
        "job_run_id": job_run_id,
        "timestamp": datetime.now().isoformat()
    }
    
    dbutils.notebook.exit(json.dumps(error_summary))

## 🔍 Como Este Notebook é Usado

### 📋 **Parâmetros Automáticos**
O DINO SDK automaticamente configura os widgets deste notebook quando cria jobs:

```python
create_dino_job(
    catalog_name="data_master_dev_dbw",
    schema_name="bronze_volumes", 
    table_name="vendas_2024",
    is_automated=True
)
```

### 🎯 **Fluxo de Execução**
1. **Configuração** - Lê parâmetros via widgets
2. **Validação** - Verifica arquivos fonte 
3. **Schema** - Detecta schema automaticamente
4. **Ingestão** - Processa todos os dados
5. **Clustering** - Aplica liquid clustering se habilitado
6. **Validação** - Confirma gravação bem-sucedida
7. **Resultado** - Retorna JSON com status

### 📊 **Monitoramento**
- Logs detalhados em cada etapa
- Métricas de performance
- Validação de integridade
- Status de execução claro

### 🔄 **Reutilização**
Este notebook é genérico e pode ser usado para qualquer ingestão configurada pelo DINO SDK.